# 06 - Sheaves: the cousin that completes the triad

Notebook 01 built *covers* and *sieves*, the ingredients that define a Grothendieck
topology (a site). A **sheaf** is what lives on that site: a presheaf that satisfies a
gluing condition over those covers. So sheaves are not a bolt-on here, they are the
conceptual sequel to 01, and the third word in this repo's title.

This notebook builds the cellular sheaf Laplacian from scratch, shows the one place it
is *morally* the right tool for our blast-radius signal, then shows why a learned sheaf
network still cannot use it on these featureless graphs, and lands the `SheafNN` at the
base rate to confirm it. The lesson is the repo's refrain: the signal, not the
architecture.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
%matplotlib inline
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.data import _flat, _segmented
from src.operators import n_components
from src.srg_data import rook44, shrikhande, make_dataset as srg_dataset
from src.sheaf import SheafNN
np.set_printoptions(precision=2, suppress=True)
plt.rcParams["figure.figsize"] = (7, 4)

## A cellular sheaf, in one function

A cellular sheaf on a graph puts a vector space (a *stalk*) `R^d` on every node and
edge, and a linear *restriction map* `B[v,e]` on each node-edge incidence. The sheaf
Laplacian `L_F` acts on the stacked node stalks (`n*d` dimensions). Its blocks are

- diagonal: `L[v,v] = sum over incident e of B[v,e]^T B[v,e]`
- off-diagonal for edge `{u,v}`: `L[u,v] = - B[u,e]^T B[v,e]`

With the trivial sheaf (`d=1`, all maps `= 1`) this is exactly the ordinary graph
Laplacian `D - A`.

In [2]:
def sheaf_laplacian(edges, n, d, B):
    """edges: list of (u, v). B(v, ei) -> d x d restriction map for node v on edge ei."""
    L = np.zeros((n * d, n * d))
    def blk(i, j):
        return (slice(i * d, (i + 1) * d), slice(j * d, (j + 1) * d))
    for ei, (u, v) in enumerate(edges):
        Bu, Bv = B(u, ei), B(v, ei)
        L[blk(u, u)] += Bu.T @ Bu
        L[blk(v, v)] += Bv.T @ Bv
        L[blk(u, v)] += -Bu.T @ Bv
        L[blk(v, u)] += -Bv.T @ Bu
    return L

def undirected_edges(A):
    S = ((A + A.T) > 0)
    return [(i, j) for i in range(A.shape[0]) for j in range(i + 1, A.shape[0]) if S[i, j]]

# trivial sheaf (d=1, identity maps) == graph Laplacian D - A
A = _flat(6)
E = undirected_edges(A)
Lf = sheaf_laplacian(E, 6, 1, lambda v, e: np.eye(1))
Asym = ((A + A.T) > 0).astype(float)
Lgraph = np.diag(Asym.sum(1)) - Asym
print("trivial sheaf Laplacian == graph Laplacian:", np.allclose(Lf, Lgraph))

trivial sheaf Laplacian == graph Laplacian: True


## The kernel of the sheaf Laplacian sees global topology

For the trivial sheaf, the dimension of the kernel of `L_F` is the number of connected
components (it is the zeroth sheaf cohomology, `H^0`). That is *exactly* our
blast-radius signal: 1 component when flat, k when segmented. So a sheaf-Laplacian
read-out is the morally correct tool for this task, more so than covers.

In [3]:
def laplacian_nullity(A):
    S = ((A + A.T) > 0).astype(float)
    L = np.diag(S.sum(1)) - S
    eig = np.linalg.eigvalsh(L)
    return int((eig < 1e-8).sum())

for name, A in [("flat (12-cycle)", _flat(12)),
                ("segmented (2x6)", _segmented(12, 2)),
                ("segmented (3x4)", _segmented(12, 3))]:
    print(f"{name:18s} ker(L) dim = {laplacian_nullity(A)}   #components = {n_components(A)}")

flat (12-cycle)    ker(L) dim = 1   #components = 1
segmented (2x6)    ker(L) dim = 2   #components = 2
segmented (3x4)    ker(L) dim = 3   #components = 3


So the information is right there in the spectrum. The catch is that a finite-layer,
feature-driven neural network does not read the kernel off the spectrum; it diffuses
features. And on these graphs there are no features to diffuse.

## Why a learned sheaf cannot use it here: the homogeneous reduction

A sheaf network learns its restriction maps from features. Our node feature is a
constant and the graphs are vertex-transitive, so there is nothing to condition on: the
learned maps come out identical at every incidence. Call that single map `B`. Then

`L_F = (D - A) (x) (B^T B) = L_graph (x) M`,

a Kronecker product. Diffusion through it is just graph diffusion, replicated across
the `d` stalk channels and mixed by `M`. A multi-channel GCN. We can check the
factorisation numerically.

In [4]:
rng = np.random.default_rng(0)
d = 3
B = rng.standard_normal((d, d))           # one shared restriction map (homogeneous)
A = _segmented(12, 2)
E = undirected_edges(A)
Lf = sheaf_laplacian(E, 12, d, lambda v, e: B)
Asym = ((A + A.T) > 0).astype(float)
Lgraph = np.diag(Asym.sum(1)) - Asym
kron = np.kron(Lgraph, B.T @ B)
print("homogeneous sheaf Laplacian == L_graph (x) (B^T B):", np.allclose(Lf, kron))

homogeneous sheaf Laplacian == L_graph (x) (B^T B): True


## So the SheafNN lands at the base rate

The repo's `SheafNN` is exactly this homogeneous sheaf diffusion with learnable maps
(more capable than the original's fixed sheaf, so this is not a strawman). We train it
on a balanced flat-vs-segmented set and on the Rook-vs-Shrikhande pair. It sits at
chance on both, while a one-line component count is perfect.

In [5]:
def gdict(A):
    n = A.shape[0]
    return {"Asym": torch.from_numpy(((A + A.T) > 0).astype(np.float32)),
            "x": torch.from_numpy(np.ones((n, 1), np.float32))}

def train_sheaf(graphs, y, tr, te, seed=0, epochs=60):
    torch.manual_seed(seed)
    m = SheafNN(in_dim=1, d=3, h=16, layers=2)
    opt = torch.optim.Adam(m.parameters(), lr=1e-2, weight_decay=1e-4)
    lossf = torch.nn.BCEWithLogitsLoss()
    yt = torch.tensor(y, dtype=torch.float32)
    for _ in range(epochs):
        m.train(); opt.zero_grad()
        logit = torch.stack([m(graphs[i]).squeeze() for i in tr])
        lossf(logit, yt[tr]).backward(); opt.step()
    m.eval()
    with torch.no_grad():
        pred = (torch.stack([m(graphs[i]).squeeze() for i in te]).sigmoid() > 0.5).long()
    return (pred == yt[te].long()).float().mean().item()

# balanced flat vs segmented across a few sizes
rng = np.random.default_rng(0)
graphs, y = [], []
for _ in range(120):
    n = int(rng.choice([12, 18, 24]))
    if rng.random() < 0.5:
        graphs.append(gdict(_flat(n))); y.append(1)
    else:
        graphs.append(gdict(_segmented(n, int(rng.choice([2, 3]))))); y.append(0)
y = np.array(y)
order = rng.permutation(len(y)); cut = 80
tr, te = order[:cut], order[cut:]
acc_fs = train_sheaf(graphs, y, tr, te)

# component-count baseline on the same test graphs
comp_pred = np.array([1 if n_components(
    (graphs[i]["Asym"].numpy())) == 1 else 0 for i in te])
acc_comp = (comp_pred == y[te]).mean()
print(f"flat vs segmented:  SheafNN acc = {acc_fs:.2f}   (chance ~0.50)")
print(f"flat vs segmented:  component-count baseline acc = {acc_comp:.2f}")

flat vs segmented:  SheafNN acc = 0.50   (chance ~0.50)
flat vs segmented:  component-count baseline acc = 1.00


In [6]:
# Rook vs Shrikhande (cospectral SRG pair)
As, ys = srg_dataset(n_per_class=100, seed=0)
g = [gdict(A) for A in As]
order = np.random.default_rng(1).permutation(len(ys)); cut = int(0.7 * len(ys))
acc_srg = train_sheaf(g, ys, order[:cut], order[cut:])
print(f"Rook vs Shrikhande: SheafNN acc = {acc_srg:.2f}   (chance ~0.50; only the sieve cover separates this)")

Rook vs Shrikhande: SheafNN acc = 0.45   (chance ~0.50; only the sieve cover separates this)


## Reading

The sheaf Laplacian's kernel literally encodes the number of components, the signal we
want, yet the learned sheaf network cannot reach it: with no features to shape the
restriction maps, it collapses to graph diffusion and lands at chance, exactly where
GCN and GIN land. The obstruction is the missing global signal, not the model.

None of this says sheaf networks are weak. Their home turf is heterophilous node
classification with real features, where the learned per-edge transport and resistance
to oversmoothing genuinely pay off. These featureless expressivity tasks are simply not
that setting. On real data too (the LANL probe, notebook 07) a trained sheaf detector
loses to a no-GNN novelty baseline, for the same reason: here the signal is not
structural.